# Intrinsic Wavefront: Zernikes Zj (v3)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-22  
**Last Modified:** 2026-04-22  
**Status:** Draft  
**Keywords:** AOS, Intrinsic Wavefront, Higher-Order Zernikes, Full Array Mode  

## Description

Extend the Z4 check (`intrinsics_checkZ4`) to every Zernike term in the stored Danish v1.0 nollIndices list (default `ZJ_LIST = [4, 5, … 19, 22, 23, 24, 25, 26]` — Z20 and Z21 are not carried by the danish fit). For each Zj:

1. Start from the Danish v1.0 FAM mktable parquet outputs for 2026-03-15 → 2026-04-09, and restrict to donuts whose visit has rotator angle near 0°. At rotator ≈ 0° the OCS and CCS field-angle frames coincide, which avoids the rotator-dependent frame confusion that appears on non-rotationally-symmetric Zernikes such as Z5/Z6.
2. Drop any `(day_obs, seq_num)` whose count of good donuts (after the rotator + matched_intra_extra cuts) is below `MIN_DONUTS_PER_VISIT` — visits with partial focal-plane coverage otherwise distort the binned maps.
3. Subtract the per-visit linear (k1, k2, k3) fit from the data Zj using the matching `_fits.parquet` file.
4. Compute two independent intrinsic Zj maps:
   - **Transposed Pipeline** — per-donut `zk_intrinsic_OCS` with the **intra-CCD fpx↔fpy swap** applied via `build_transposed_pipeline_zk`. The plain pipeline intrinsic carries the same axis swap diagnosed for Z4 in `intrinsics_checkZ4`, so we use the transposed version here as the pipeline comparison for all Zj.
   - **Batoid v2** — `batoid_rubin.LSSTBuilder.build_det` + `batoid.zernike` per CCD, following the recipe from `Intrinsic Zk.ipynb`. Includes the per-CCD focal-plane height map via `LSSTBuilder.ccd_height_map_dir`.
5. Produce one 2×3 focal-plane page per Zj: Data (Zj − linear), Transposed Pipeline intrinsic, Batoid v2 intrinsic, Data−Transposed Pipeline, Data−Batoid v2, Transposed Pipeline−Batoid v2. Colour scales are grouped: the three top panels (Data + two intrinsics) share one `(vmin, vmax)` from the per-map 3 %/97 % percentiles; the two Data−Intrinsic diffs share another; the Transposed Pipeline−Batoid v2 panel gets its own.

**Output:** In-notebook plots and one multi-page PDF at `PDF_OUTPUT` (rotator histogram + Batoid v2 axis-convention validation + one page per Zj).

**Based on:**
- `aos/intrinsics_checkZ4.ipynb` — per-CCD Batoid v2, transposed-pipeline diagnostic, linear-fit handling
- `~/Astrophysics/Code/Claude/Intrinsic Zk.ipynb` — batoid_rubin per-CCD recipe
- `intrinsics_mktable.ipynb` / `code/intrinsics_lib.py` — streaming parquet donut/visits format
- `code/dz_fitting.py` — focal-plane Zernike basis for the per-visit linear fit

<a id='changelog'></a>
## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-22 | Aaron Roodman | Initial version — Z5 … Z17, rotator ≈ 0° slice, Pipeline vs Batoid v2 |
| 2026-04-22 | Aaron Roodman | v2: switched input from HDF5 to Danish v1.0 parquet (donuts + sidecar visits + fits). Added `MIN_DONUTS_PER_VISIT` cut. Per-group 3 %/97 % colour scales. Axis-convention validation cell. Fixed double `np.rad2deg` in the kept-visits summary. |
| 2026-04-22 | Aaron Roodman | v3: extended `ZJ_LIST` to the full stored list Z4 … Z19 + Z22 … Z26; switched pipeline comparison from plain Pipeline to **Transposed Pipeline** (per-CCD fpx↔fpy swap) via `build_transposed_pipeline_zk`. Added Batoid v2 NaN diagnostic (overall rate, per-CCD table, focal-plane scatter). |
| 2026-04-22 | Aaron Roodman | v3 tweak: removed the half-cell inset in `_per_ccd_field_angle_grid` so the grid spans each CCD corner-to-corner; bumped `INTRINSIC_PER_CCD_GRID_N` from 8 to 12. Batoid v2 now runs on `LSST_i.yaml` with λ = 756 nm (i-band centre, matches the data). Wrapped `batoid.zernike` in try/except so vignetted grid points (RuntimeError: "Rank 0 matrix") at the extreme field corners of the outer rafts are skipped cleanly, and the interpolator is built from the surviving grid points. Detectors with fewer than 4 good points are dropped entirely; their donuts read NaN via the default fill. |
| 2026-04-22 | Aaron Roodman | v3 tweak: changed the default `BATOID_V2_YAML` from the baseline design `LSST_i.yaml` to the as-built `Rubin_v3.14_i.yaml`. Override the parameter to compare. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Input donuts parquet (Danish v1.0 results). The streaming mktable writer
# now uses parquet (list<float> for the Zernike arrays) with a sidecar
# visits parquet at {stem}_visits.parquet.
INPUT_DONUTS = 'output/aos_fam_danish_v1_triplets_20260315_20260409.parquet'

# Linear-fit parquet (auto-derived as {stem}_fits.parquet if None)
FIT_PARQUET = None

# Coordinate system used for the linear fit (must match the fit parquet)
COORD_SYS = 'OCS'

# Focal-plane radius for the linear-fit basis normalization (degrees),
# matches code/dz_fitting.py: focal_plane_zernike_basis default.
FP_RADIUS_DEG = 1.75

# Zernike Noll indices to plot. One PDF page per Zj. Default covers
# Z4 through Z26, skipping Z20 and Z21 (which are not in the stored
# danish nollIndices list).
ZJ_LIST = list(range(4, 20)) + [22, 23, 24, 25, 26]

# Rotator-angle selection (degrees). Only donuts whose visit has
# rotator_angle within ROTATOR_TOL_DEG of ROTATOR_CENTER_DEG are kept.
# At rotator ≈ 0° the OCS and CCS field-angle frames are aligned, so the
# OCS-stored data can be compared directly against the Batoid v2 intrinsic
# evaluated in CCS.
ROTATOR_CENTER_DEG = 0.0
ROTATOR_TOL_DEG    = 5.0

# Per-visit quality cut: drop any (day_obs, seq_num) whose number of
# good donuts (after the rotator + matched_intra_extra cuts) is below
# MIN_DONUTS_PER_VISIT. Partial-coverage or failed visits otherwise
# distort the focal-plane maps.
MIN_DONUTS_PER_VISIT = 500

# Per-CCD intrinsic grid (Batoid v2). grid_n × grid_n points spanning
# each CCD's CCS field-angle footprint corner-to-corner (no inset), so the
# LinearNDInterpolator's convex hull covers the whole CCD and donuts near
# the edges no longer fall outside and come back NaN.
INTRINSIC_PER_CCD_GRID_N = 12

# Batoid v2 config. Almost all the FAM data is in i-band, so use the
# i-band fiducial optic and a representative i-band wavelength.
# LSST SRD effective wavelengths (nm): u=367, g=482, r=622, i=755,
# z=869, y=971.
#
# Two fiducial optic yamls are available:
#   'LSST_i.yaml'         — the baseline LSST telescope+camera design
#   'Rubin_v3.14_i.yaml'  — the as-built Rubin v3.14 design (default)
# Override BATOID_V2_YAML to compare.
BATOID_V2_YAML         = 'Rubin_v3.14_i.yaml'
BATOID_V2_WAVELENGTH_M = 756e-9
BATOID_V2_JMAX         = 28
BATOID_V2_EPS          = 0.612
BATOID_V2_NX           = 63

# Writable data directory for batoid_rubin (FEA / bending / CCD height map
# Zenodo datasets). Setup cell sets BATOID_RUBIN_DATA_DIR only if not
# already present in the shell environment.
BATOID_RUBIN_DATA_DIR = '~/batoid_rubin_data'

# Focal-plane map binning
FP_NSTEPS = 257
FP_MAX_MM = 320.0

# Restrict to donuts whose intra and extra centroids matched
REQUIRE_MATCHED = True

# Per-group colour-scale percentiles for the results 2×3 grid.
# vmin = min over group of the low percentile; vmax = max over group of
# the high percentile.  Groups: (Data, Pipeline, Batoid v2) share one
# scale, (Data−Pipeline, Data−Batoid v2) share another, Pipeline−Batoid v2
# gets its own.
PLOT_PCTL_LOW  = 3.0
PLOT_PCTL_HIGH = 97.0

# Output PDF (one page per Zj)
PDF_OUTPUT = 'output/intrinsic_Zj.pdf'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits import axes_grid1

from scipy.stats import binned_statistic_2d
from scipy.interpolate import LinearNDInterpolator
import tqdm

from lsst.obs.lsst import LsstCam
from lsst.afw.cameraGeom import FIELD_ANGLE

# Writable batoid_rubin data dir — the default under the installed conda env
# is read-only and the first LSSTBuilder call downloads FEA / bending /
# ccd_height_map datasets from Zenodo. Respect any user-set env var.
os.environ.setdefault(
    'BATOID_RUBIN_DATA_DIR',
    os.path.expanduser(BATOID_RUBIN_DATA_DIR),
)
Path(os.environ['BATOID_RUBIN_DATA_DIR']).mkdir(parents=True, exist_ok=True)
print(f"BATOID_RUBIN_DATA_DIR = {os.environ['BATOID_RUBIN_DATA_DIR']}")

import batoid
from batoid_rubin import LSSTBuilder

# Shared cameraGeom helpers (common/camera_utils.py)
sys.path.insert(0, str(Path.cwd().parent))
from common.camera_utils import pixel_to_focal  # noqa: E402

plt.rcParams['figure.dpi'] = 110

<a id='functions'></a>
## Helper Functions

In [ ]:
def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Attach a vertical colorbar matched to the host axes' height."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1.0 / aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes('right', size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


# ------------------------------------------------------------------
# Donut → focal-plane coordinates
# ------------------------------------------------------------------

def compute_fp_coords(aosTable, camera, x_col='centroid_x_intra',
                      y_col='centroid_y_intra'):
    """Evaluate fpx/fpy in mm for every donut via the per-detector
    cameraGeom PIXELS → FOCAL_PLANE transform (vectorised per sensor).
    Works with either an astropy QTable or a pandas DataFrame.
    """
    det_names = np.asarray(aosTable['detector']).astype(str)
    x_pix = np.asarray(aosTable[x_col], dtype=float)
    y_pix = np.asarray(aosTable[y_col], dtype=float)
    fpx = np.full_like(x_pix, np.nan)
    fpy = np.full_like(y_pix, np.nan)
    for det in camera:
        name = det.getName()
        mask = (det_names == name)
        if not np.any(mask):
            continue
        fx, fy = pixel_to_focal(x_pix[mask], y_pix[mask], det)
        fpx[mask] = fx
        fpy[mask] = fy
    return fpx, fpy


def ccd_center_fp(camera):
    """Return {detector_name: (cx_mm, cy_mm)} — each CCD's bbox-center
    focal-plane coordinate, used by the transposed-pipeline swap."""
    centers = {}
    for det in camera:
        c = det.getBBox().getCenter()
        cx, cy = pixel_to_focal(np.array([c.getX()]),
                                np.array([c.getY()]), det)
        centers[det.getName()] = (float(cx[0]), float(cy[0]))
    return centers


def _per_ccd_field_angle_grid(det, grid_n):
    """n×n CCS field-angle grid (radians) spanning one CCD's footprint
    corner-to-corner (no inset).

    Uses the same DVCS→CCS transpose as `Intrinsic Zk.ipynb`:
    CCS x = DVCS y of the corners, CCS y = DVCS x. Unlike that notebook,
    grid points include the corner values directly (``np.linspace(xmin,
    xmax, grid_n)``), so the resulting `LinearNDInterpolator` convex hull
    covers the full CCD footprint and donuts near the edges are not
    extrapolated to NaN.
    """
    corners = det.getCorners(FIELD_ANGLE)
    xmin = min(c.y for c in corners)
    xmax = max(c.y for c in corners)
    ymin = min(c.x for c in corners)
    ymax = max(c.x for c in corners)
    xs = np.linspace(xmin, xmax, grid_n)
    ys = np.linspace(ymin, ymax, grid_n)
    xx, yy = np.meshgrid(xs, ys)
    return xx.ravel(), yy.ravel()


# ------------------------------------------------------------------
# Transposed pipeline intrinsic (x ↔ y per CCD, multi-Zj diagnostic)
# ------------------------------------------------------------------

def build_transposed_pipeline_zk(fpx_mm, fpy_mm, det_names,
                                 Zk_pipeline_intrinsic_um,
                                 camera, keep_mask=None):
    """Per-CCD x↔y swap of the pipeline intrinsic, generalised to all Zj.

    Multi-Zj analogue of the Z4 notebook's `build_transposed_pipeline_z4`.
    For each donut compute its offset `(dx, dy)` from its CCD's bbox
    center in focal-plane mm, swap to `(dy, dx)`, then read the pipeline
    intrinsic for every Zj at the swapped position via
    `scipy.spatial.cKDTree` nearest-neighbor over the kept donuts.

    Parameters
    ----------
    fpx_mm, fpy_mm : ndarray (n_donuts,)
    det_names : ndarray of str (n_donuts,)
    Zk_pipeline_intrinsic_um : ndarray (n_donuts, n_Z) — pipeline zk_intrinsic
    camera : lsst.afw.cameraGeom.Camera
    keep_mask : optional bool mask of donuts to use as the NN reference set

    Returns
    -------
    Zk_transposed : ndarray (n_donuts, n_Z)  transposed pipeline intrinsic
    fpx_swap, fpy_swap : ndarray (n_donuts,)
    det_centers : dict {det_name: (cx_mm, cy_mm)}
    """
    from scipy.spatial import cKDTree
    det_centers = ccd_center_fp(camera)
    fpx_swap = np.full_like(fpx_mm, np.nan)
    fpy_swap = np.full_like(fpy_mm, np.nan)
    for name, (cx, cy) in det_centers.items():
        mask = (det_names == name)
        if not np.any(mask):
            continue
        dx = fpx_mm[mask] - cx
        dy = fpy_mm[mask] - cy
        fpx_swap[mask] = cx + dy
        fpy_swap[mask] = cy + dx
    if keep_mask is None:
        keep_mask = np.ones(len(fpx_mm), dtype=bool)
    ref_pts = np.column_stack([fpx_mm[keep_mask], fpy_mm[keep_mask]])
    ref_vals = Zk_pipeline_intrinsic_um[keep_mask]      # (n_kept, n_Z)
    tree = cKDTree(ref_pts)
    _, nn_idx = tree.query(np.column_stack([fpx_swap, fpy_swap]))
    return ref_vals[nn_idx], fpx_swap, fpy_swap, det_centers


# ------------------------------------------------------------------
# Batoid v2 multi-Zj intrinsic
# ------------------------------------------------------------------

def build_zk_intrinsic_per_ccd_batoid_v2(camera, zj_list,
                                         yaml_name=BATOID_V2_YAML,
                                         wavelength_m=BATOID_V2_WAVELENGTH_M,
                                         grid_n=INTRINSIC_PER_CCD_GRID_N,
                                         jmax=BATOID_V2_JMAX,
                                         eps=BATOID_V2_EPS,
                                         nx=BATOID_V2_NX,
                                         verbose=True):
    """Per-CCD Batoid v2 intrinsic for a list of Zernike indices.

    Mirrors the Z4 notebook's `build_z4_intrinsic_per_ccd_batoid_v2`, but
    evaluates ``batoid.zernike`` once per grid point (it returns all Zk up
    to ``jmax`` in one call) and builds a **vector-valued**
    ``LinearNDInterpolator`` per CCD whose values have shape
    ``(n_pts, len(zj_list))``.

    At the far corners of the outer rafts the rays can be fully vignetted,
    in which case ``batoid.zernike`` raises a ``RuntimeError`` ("Rank 0
    matrix…"). Those grid points are caught and dropped before the
    interpolator is built; if fewer than 4 grid points survive for a given
    CCD the detector is skipped entirely (its donuts will read NaN via
    ``evaluate_zk_per_ccd_multi``'s default fill).

    Returns
    -------
    interp_dict : dict {det_name: LinearNDInterpolator}
        Query each with (thx_deg, thy_deg); result is (n_query, n_zj) in μm.
    """
    zj_list = list(zj_list)
    if max(zj_list) > jmax:
        raise ValueError(f'jmax={jmax} too small for zj_list={zj_list}')

    fiducial = batoid.Optic.fromYaml(yaml_name)
    builder = LSSTBuilder(
        fiducial, dof_coord_system='OCS',
        flip_m2_bending_modes=False, dof_angle_units='degree',
    )
    scale_um = wavelength_m * 1.0e6   # waves × λ(m) × 1e6 = μm

    interp_dict = {}
    it = camera
    if verbose:
        print(f'batoid_rubin v2 multi-Zj: {grid_n}×{grid_n} grid, '
              f'zj_list={zj_list}, yaml={yaml_name}, '
              f'λ={wavelength_m*1e9:.1f} nm')
        it = tqdm.tqdm(list(camera), desc='batoid v2 per CCD')

    n_failed_gp_total   = 0
    n_skipped_detectors = 0
    for det in it:
        telescope = builder.build_det(det.getId())
        thx_rad, thy_rad = _per_ccd_field_angle_grid(det, grid_n)
        values_um = np.full((len(thx_rad), len(zj_list)), np.nan)
        for i, (x, y) in enumerate(zip(thx_rad, thy_rad)):
            try:
                zk_all = batoid.zernike(
                    telescope, theta_x=float(x), theta_y=float(y),
                    wavelength=wavelength_m, projection='gnomonic',
                    jmax=jmax, eps=eps, nx=nx,
                )
            except RuntimeError:
                # batoid.zernike raises RuntimeError (rank-deficient fit)
                # when all rays at this field angle are vignetted by the
                # optics. Leave the row as NaN and drop it below.
                continue
            for k, zj in enumerate(zj_list):
                values_um[i, k] = zk_all[zj] * scale_um

        good = np.isfinite(values_um).all(axis=1)
        n_failed = int((~good).sum())
        n_failed_gp_total += n_failed
        if good.sum() < 4:
            if verbose:
                print(f'  {det.getName()}: SKIPPED '
                      f'({good.sum()}/{len(thx_rad)} grid pts survived)')
            n_skipped_detectors += 1
            continue

        thx_deg = np.rad2deg(thx_rad[good])
        thy_deg = np.rad2deg(thy_rad[good])
        pts = np.column_stack([thx_deg, thy_deg])
        interp_dict[det.getName()] = LinearNDInterpolator(
            pts, values_um[good], fill_value=np.nan,
        )

    if verbose:
        print(f'Batoid v2 build summary: '
              f'{n_failed_gp_total} vignetted grid points total, '
              f'{n_skipped_detectors} detectors skipped entirely.')
    return interp_dict


def evaluate_zk_per_ccd_multi(det_names, thx_deg, thy_deg, interp_dict, n_zj):
    """Dispatch each donut to its CCD's multi-Zj interpolator.
    Returns (n_donuts, n_zj) array in μm."""
    out = np.full((len(det_names), n_zj), np.nan, dtype=float)
    for name, interp in interp_dict.items():
        mask = (det_names == name)
        if not np.any(mask):
            continue
        out[mask] = interp(thx_deg[mask], thy_deg[mask])
    return out


# ------------------------------------------------------------------
# Linear (k1, k2, k3) basis evaluation (same basis as dz_fitting.py)
# ------------------------------------------------------------------

def linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG):
    """k1 + 2·k2·(thx_deg/R) + 2·k3·(thy_deg/R) in μm.
    Matches `code/dz_fitting.focal_plane_zernike_basis` (Z1 piston, Z2 tilt,
    Z3 tip on the unit disk).
    """
    x = thx_deg / fp_radius
    y = thy_deg / fp_radius
    return k1 + 2.0 * k2 * x + 2.0 * k3 * y


# ------------------------------------------------------------------
# Binning + plotting
# ------------------------------------------------------------------

def bin_median_2d(xval, yval, zval, xbins, ybins):
    """Median of zval on an (xbins × ybins) 2D grid over (xval, yval)."""
    stat, x_edges, y_edges, _ = binned_statistic_2d(
        xval, yval, zval, statistic='median', bins=[xbins, ybins]
    )
    return stat, x_edges, y_edges


def _group_limits(stats, q_lo=PLOT_PCTL_LOW, q_hi=PLOT_PCTL_HIGH):
    """Per-group (vmin, vmax): for each map in the group take its (q_lo, q_hi)
    percentiles; combine as vmin = min across maps of the low percentile,
    vmax = max across maps of the high percentile. Outer envelope, asymmetric.
    """
    lows, highs = [], []
    for s in stats:
        flat = s.ravel()
        flat = flat[np.isfinite(flat)]
        if flat.size == 0:
            continue
        lo, hi = np.percentile(flat, [q_lo, q_hi])
        lows.append(lo)
        highs.append(hi)
    if not lows or not highs:
        return -1.0, 1.0
    return float(min(lows)), float(max(highs))


def plot_fp_grid(maps, xe, ye, title, cmap='RdBu_r',
                 groups=((0, 1, 2), (3, 4), (5,)),
                 q_lo=PLOT_PCTL_LOW, q_hi=PLOT_PCTL_HIGH):
    """Render six focal-plane maps on a 2×3 grid with grouped colour scales.

    `groups` partitions the six panels into colour-scale groups (defaults
    match the Data / Data−Intrinsic / Intrinsic−Intrinsic layout used here).
    Each group's (vmin, vmax) comes from per-map (q_lo, q_hi) percentiles
    via `_group_limits`; all panels use the same diverging cmap.
    """
    assert len(maps) == 6
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))

    # Assign (vmin, vmax) per panel from its group
    idx_lim = [None] * 6
    for group in groups:
        stats = [maps[gi][1] for gi in group]
        vmin, vmax = _group_limits(stats, q_lo=q_lo, q_hi=q_hi)
        for gi in group:
            idx_lim[gi] = (vmin, vmax)

    for idx, (subtitle, stat) in enumerate(maps):
        ax = axes.flat[idx]
        vmin, vmax = idx_lim[idx]
        im = ax.imshow(
            stat.T, origin='lower',
            extent=[xe[0], xe[-1], ye[0], ye[-1]],
            cmap=cmap, interpolation='none', aspect='equal',
            vmin=vmin, vmax=vmax,
        )
        ax.set_xlabel('fpx [mm]')
        ax.set_ylabel('fpy [mm]')
        ax.set_title(subtitle)
        add_colorbar(im).set_label('Z [μm]')
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    return fig

<a id='data'></a>
## Data Access

In [ ]:
# Danish v1.0 parquet inputs:
#   donuts parquet      = INPUT_DONUTS
#   visits parquet      = {stem}_visits.parquet   (sidecar written by mktable)
#   fits  parquet       = {stem}_fits.parquet     (or FIT_PARQUET if set)
donuts_path = Path(INPUT_DONUTS)
if not donuts_path.exists():
    raise FileNotFoundError(f'Donuts parquet not found: {donuts_path}')

visits_path = donuts_path.parent / f'{donuts_path.stem}_visits.parquet'
if not visits_path.exists():
    raise FileNotFoundError(f'Visits parquet not found: {visits_path}')

if FIT_PARQUET is None:
    fits_path = donuts_path.parent / f'{donuts_path.stem}_fits.parquet'
else:
    fits_path = Path(FIT_PARQUET)
if not fits_path.exists():
    raise FileNotFoundError(f'Fit parquet not found: {fits_path}')

print(f'Donuts:      {donuts_path}')
print(f'Visits:      {visits_path}')
print(f'Fit parquet: {fits_path}')

aosTable   = pd.read_parquet(donuts_path)
visit_info = pd.read_parquet(visits_path)
fits_table = pd.read_parquet(fits_path)
print(f'Loaded {len(aosTable):,} donuts, {len(aosTable.columns)} columns, '
      f'{len(visit_info)} visits')
print(f'Fit rows: {len(fits_table)}, columns: {len(fits_table.columns)}')

In [ ]:
# Build the per-donut keep mask in three stages:
#   1) Rotator window: |rotator_angle − ROTATOR_CENTER_DEG| ≤ ROTATOR_TOL_DEG
#   2) Matched intra/extra (optional)
#   3) Per-visit donut count ≥ MIN_DONUTS_PER_VISIT after (1)+(2)

# 1) Rotator. The Danish v1.0 parquet keeps rotator_angle in the visits
#    sidecar rather than duplicated per donut, so we look it up via a
#    (day_obs, seq_num) merge. Fall back to a per-donut column if present.
if 'rotator_angle' in aosTable.columns:
    rot = np.asarray(aosTable['rotator_angle'], dtype=float)
    rot_source = 'donuts'
else:
    if 'rotator_angle' not in visit_info.columns:
        raise KeyError("Neither the donuts parquet nor the visits parquet "
                       "carries a 'rotator_angle' column.")
    rot_lookup = visit_info[['day_obs', 'seq_num', 'rotator_angle']]
    donuts_key = aosTable[['day_obs', 'seq_num']]
    merged_rot = donuts_key.merge(rot_lookup, on=['day_obs', 'seq_num'],
                                  how='left')
    rot = merged_rot['rotator_angle'].to_numpy(dtype=float)
    rot_source = 'visits'
print(f"rotator_angle source: {rot_source} parquet")

# Sanity-check units and auto-convert radians → degrees if the range is tiny.
if np.all(np.abs(rot[np.isfinite(rot)]) <= 2 * np.pi + 0.1):
    print('rotator_angle appears to be in radians; converting to degrees.')
    rot = np.rad2deg(rot)
rmin, rmax = float(np.nanmin(rot)), float(np.nanmax(rot))
print(f'rotator_angle range [deg]: [{rmin:+.2f}, {rmax:+.2f}]')

rot_mask = np.abs(rot - ROTATOR_CENTER_DEG) <= ROTATOR_TOL_DEG
# Treat NaN rotator as a fail.
rot_mask &= np.isfinite(rot)
print(f'Rotator cut |rot − {ROTATOR_CENTER_DEG:.1f}°| ≤ {ROTATOR_TOL_DEG:.1f}°: '
      f'{int(rot_mask.sum()):,}/{len(rot_mask):,} donuts')

# 2) Matched intra/extra.
keep = rot_mask.copy()
if REQUIRE_MATCHED and 'matched_intra_extra' in aosTable.columns:
    keep &= np.asarray(aosTable['matched_intra_extra'], dtype=bool)
print(f'After matched_intra_extra cut: {int(keep.sum()):,} donuts kept')

# 3) Per-visit good-donut count threshold.
day_obs_arr = np.asarray(aosTable['day_obs'], dtype=np.int64)
seq_num_arr = np.asarray(aosTable['seq_num'], dtype=np.int64)
_visit_id = day_obs_arr.astype(np.int64) * 100000 + seq_num_arr.astype(np.int64)
uniq_visits, counts = np.unique(_visit_id[keep], return_counts=True)
good_visits = set(uniq_visits[counts >= MIN_DONUTS_PER_VISIT].tolist())
n_total_visits_seen = len(uniq_visits)
n_good_visits = len(good_visits)
print(f'Visits with ≥ {MIN_DONUTS_PER_VISIT} good donuts: '
      f'{n_good_visits}/{n_total_visits_seen}')
visit_ok = np.fromiter((vid in good_visits for vid in _visit_id),
                       count=len(_visit_id), dtype=bool)
keep &= visit_ok
print(f'After per-visit count cut: {int(keep.sum()):,} donuts kept '
      f'across {n_good_visits} visits')

In [ ]:
# Summary of every (day_obs, seq_num) that survives all three cuts and
# thus contributes to the focal-plane maps below. Reuses the per-donut
# `rot` array from the filter cell (already normalised to degrees there),
# so there is exactly one units conversion in the notebook — re-deriving
# the rotator angle from visit_info here would re-apply the radians→deg
# heuristic and spuriously multiply small degree values by 180/π.
_rdf = pd.DataFrame({
    'day_obs':           day_obs_arr[keep],
    'seq_num':           seq_num_arr[keep],
    'rotator_angle_deg': rot[keep],
})
summary_df = (_rdf
              .groupby(['day_obs', 'seq_num'], as_index=False)
              .agg(rotator_angle_deg=('rotator_angle_deg', 'first'),
                   n_donuts=('rotator_angle_deg', 'size'))
              .sort_values(by=['day_obs', 'seq_num'])
              .reset_index(drop=True))

print(f'{len(summary_df)} visits used in the plots '
      f'(|rot − {ROTATOR_CENTER_DEG:.1f}°| ≤ {ROTATOR_TOL_DEG:.1f}°, '
      f'≥ {MIN_DONUTS_PER_VISIT} good donuts):')
with pd.option_context('display.max_rows', None,
                       'display.width', 120,
                       'display.float_format', '{:+.3f}'.format):
    print(summary_df.to_string(index=False))

_rv = summary_df['rotator_angle_deg'].to_numpy(dtype=float)
_rv = _rv[np.isfinite(_rv)]
print(f'\nRotator stats over {len(_rv)} visits [deg]: '
      f'mean={np.mean(_rv):+.3f},  std={np.std(_rv):.3f},  '
      f'abs max={np.max(np.abs(_rv)):.3f}')

<a id='analysis'></a>
## Analysis

In [ ]:
# Noll index bookkeeping for the stored zk columns.
zk_col      = f'zk_{COORD_SYS}'
zk_intr_col = f'zk_intrinsic_{COORD_SYS}'
thx_col     = f'thx_{COORD_SYS}'
thy_col     = f'thy_{COORD_SYS}'

# zk_OCS / zk_intrinsic_OCS come from parquet list<float> columns, one
# array per donut. np.stack(...tolist()) gives a (N, n_Z) float array.
zk_data_full = np.stack(aosTable[zk_col].to_list())          # (N, n_Z), μm
zk_intr_full = np.stack(aosTable[zk_intr_col].to_list())     # (N, n_Z), μm
nZk = zk_data_full.shape[1]

if 'nollIndices' in visit_info.columns:
    iZs_all = [int(n) for n in visit_info['nollIndices'].iloc[0]]
    if len(iZs_all) != nZk:
        iZs_all = list(range(4, 4 + nZk))
else:
    iZs_all = list(range(4, 4 + nZk))
iZidx_all = {iZ: i for i, iZ in enumerate(iZs_all)}
print(f'Zernike Noll indices in parquet: {iZs_all}')

missing = [j for j in ZJ_LIST if j not in iZidx_all]
if missing:
    raise KeyError(f'Requested Zj not present in data: {missing}')
print(f'Requested Zj to plot: {ZJ_LIST}')

# Field angles (deg) in the fit coordinate system.
thx_deg = np.rad2deg(np.asarray(aosTable[thx_col], dtype=float))
thy_deg = np.rad2deg(np.asarray(aosTable[thy_col], dtype=float))

# Focal-plane (fpx, fpy) in mm from the intra-focal pixel centroid via
# per-detector cameraGeom PIXELS → FOCAL_PLANE transforms.
camera = LsstCam.getCamera()
fpx, fpy = compute_fp_coords(aosTable, camera,
                             x_col='centroid_x_intra',
                             y_col='centroid_y_intra')
print(f'fpx range [mm]: [{np.nanmin(fpx):+.1f}, {np.nanmax(fpx):+.1f}]; '
      f'fpy range [mm]: [{np.nanmin(fpy):+.1f}, {np.nanmax(fpy):+.1f}]')

det_names_all = np.asarray(aosTable['detector']).astype(str)

In [ ]:
# Subtract the per-visit linear (k1, k2, k3) fit from the data Zj. The fits
# parquet carries z1toz3_z{j}_c{1,2,3} columns for each Zj in the iZs list.
day_obs = np.asarray(aosTable['day_obs'], dtype=int)
seq_num = np.asarray(aosTable['seq_num'], dtype=int)
donuts_df = pd.DataFrame({'day_obs': day_obs, 'seq_num': seq_num})

Zj_data_linear_um = {}   # {j: ndarray of shape (N,)}  data − linear fit (μm)
for j in ZJ_LIST:
    c1, c2, c3 = f'z1toz3_z{j}_c1', f'z1toz3_z{j}_c2', f'z1toz3_z{j}_c3'
    missing = [c for c in (c1, c2, c3) if c not in fits_table.columns]
    if missing:
        raise KeyError(f'Missing fit columns for Z{j}: {missing}')
    keys_j = fits_table[['day_obs', 'seq_num', c1, c2, c3]].copy()
    merged = donuts_df.merge(keys_j, on=['day_obs', 'seq_num'], how='left')
    k1 = merged[c1].to_numpy()
    k2 = merged[c2].to_numpy()
    k3 = merged[c3].to_numpy()
    Zj_linear = linear_fit_values(thx_deg, thy_deg, k1, k2, k3,
                                   fp_radius=FP_RADIUS_DEG)
    col_idx = iZidx_all[j]
    Zj_data_linear_um[j] = zk_data_full[:, col_idx] - Zj_linear

# Transposed Pipeline intrinsic. The plain pipeline zk_intrinsic shows an
# intra-CCD fpx↔fpy swap (documented in the Z4 notebook); the same swap
# appears for Z5…Z26, so we apply the per-CCD transpose here and use the
# transposed intrinsic as the Pipeline comparison below. `ccd_center_fp`
# places the per-CCD bbox center in the focal plane; the swap reflects each
# donut about that center and `scipy.spatial.cKDTree` reads the pipeline
# intrinsic at the swapped position from the kept donuts.
det_names_all = np.asarray(aosTable['detector']).astype(str)
Zk_pipeT_full, _fpx_swap, _fpy_swap, _det_centers = \
    build_transposed_pipeline_zk(
        fpx, fpy, det_names_all, zk_intr_full, camera, keep_mask=keep,
    )
# Plain pipeline (for reference) and the per-Zj transposed version.
Zj_pipe_um       = {j: zk_intr_full[:, iZidx_all[j]] for j in ZJ_LIST}
Zj_pipe_trans_um = {j: Zk_pipeT_full[:, iZidx_all[j]] for j in ZJ_LIST}

print('Per-donut statistics after linear correction '
      '(kept donuts, μm, using Transposed Pipeline):')
print(f"  {'Zj':>4s}  {'mean_data':>11s}  {'std_data':>10s}  "
      f"{'mean_pipeT':>11s}  {'std_pipeT':>10s}")
for j in ZJ_LIST:
    ad = Zj_data_linear_um[j][keep]
    ap = Zj_pipe_trans_um[j][keep]
    print(f'  Z{j:<3d}  {np.nanmean(ad):+11.4f}  {np.nanstd(ad):10.4f}  '
          f'{np.nanmean(ap):+11.4f}  {np.nanstd(ap):10.4f}')

In [ ]:
# Batoid v2 per-CCD multi-Zj intrinsic. One LSSTBuilder and one
# LinearNDInterpolator per detector; each interpolator returns
# (n_query, len(ZJ_LIST)) in μm.
interp_batoidv2 = build_zk_intrinsic_per_ccd_batoid_v2(
    camera, ZJ_LIST, grid_n=INTRINSIC_PER_CCD_GRID_N,
)
Zj_bv2_matrix = evaluate_zk_per_ccd_multi(
    det_names_all, thx_deg, thy_deg, interp_batoidv2, len(ZJ_LIST),
)   # (N, len(ZJ_LIST))
Zj_bv2_um = {j: Zj_bv2_matrix[:, k] for k, j in enumerate(ZJ_LIST)}

print('Batoid v2 intrinsic per-donut statistics (kept donuts, μm):')
for j in ZJ_LIST:
    a = Zj_bv2_um[j][keep]
    print(f'  Z{j:<3d}  mean={np.nanmean(a):+11.4f}  std={np.nanstd(a):10.4f}')

<a id='results'></a>
## Results & Plots

In [ ]:
# Common binning and PDF figure collector. Re-running this cell starts a
# fresh collection. `_collect` returns None so cells don't render a second
# inline copy of the figure.
xbins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)
ybins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)

xv = fpx[keep]
yv = fpy[keep]

_pdf_figs = []


def _collect(fig):
    _pdf_figs.append(fig)

In [ ]:
# Quality check: histogram the rotator angle across the kept visits. The
# Batoid v2 intrinsic is evaluated at exactly rotator = 0.0° (no dof
# perturbation), so any rotator drift in this sample is a direct source
# of disagreement between the data and Batoid v2 for non-rotationally-
# symmetric Zernikes (Z5, Z6, …). First page of the PDF.
_rot_visits = summary_df['rotator_angle_deg'].to_numpy(dtype=float)
_rot_visits = _rot_visits[np.isfinite(_rot_visits)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(_rot_visits, bins=40, histtype='step')
ax.axvline(0.0, color='red', lw=0.8,
           label='Batoid v2 reference (rotator = 0°)')
ax.axvspan(ROTATOR_CENTER_DEG - ROTATOR_TOL_DEG,
           ROTATOR_CENTER_DEG + ROTATOR_TOL_DEG,
           color='gray', alpha=0.1,
           label=f'rotator window ±{ROTATOR_TOL_DEG:.1f}°')
ax.set_xlabel('visit rotator_angle [deg]')
ax.set_ylabel('visits')
ax.set_title(
    f'Rotator angle over {len(_rot_visits)} kept visits    '
    f'mean={np.mean(_rot_visits):+.2f}°,  '
    f'std={np.std(_rot_visits):.2f}°,  '
    f'abs max={np.max(np.abs(_rot_visits)):.2f}°'
)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
_collect(fig)

In [ ]:
# ------------------------------------------------------------------
# Validation: does Batoid v2 per-CCD interpolator get x↔y right?
# ------------------------------------------------------------------
# The per-CCD grid is constructed with the `Intrinsic Zk.ipynb` DVCS→CCS
# transpose:  grid θx = CCS x (= DVCS y),  grid θy = CCS y (= DVCS x),
# and batoid.zernike is called with (theta_x=CCS_x, theta_y=CCS_y). The
# interpolator is therefore keyed on (CCS_x_deg, CCS_y_deg).
#
# At rotator = 0, OCS coincides with CCS — but only if the mktable stored
# thx_OCS with the CCS labelling (thx_OCS = CCS_x = DVCS_y). If instead
# thx_OCS follows the DVCS labelling (thx_OCS = DVCS_x = CCS_y), the
# interpolator is queried with x and y swapped **within each CCD**. Global
# CCD placement would still look right (a donut on a given CCD still lands
# somewhere on that CCD's grid), but the intra-CCD Zernike pattern would
# be reflected along the diagonal — exactly the kind of artifact one would
# expect in Z5/Z6 maps.
#
# Two checks:
#   (A) Numerical — for one kept donut on each of a few CCDs, compute
#       FIELD_ANGLE (DVCS, from the pixel centroid via cameraGeom) and
#       compare against the stored (thx_OCS, thy_OCS). Whichever pairing
#       matches better tells us the labelling.
#   (B) Visual    — overlay CCD corners, the 8×8 grid, and the donut
#       (thx_OCS, thy_OCS) positions for a handful of CCDs. All three
#       must overlap; if donuts land on a reflected footprint, the
#       axes are swapped.
from lsst.afw.cameraGeom import PIXELS

_val_names = ['R22_S11', 'R22_S00', 'R02_S21', 'R42_S01', 'R20_S12']
_det_map = {d.getName(): d for d in camera}
_val_names = [n for n in _val_names if n in _det_map]
if not _val_names:
    _val_names = [d.getName() for d in list(camera)[:5]]

# ---- Check (A): numerical convention ----------------------------------
print('Check (A): stored (thx_OCS, thy_OCS) vs cameraGeom FIELD_ANGLE '
      '(DVCS) for one kept donut per sampled CCD.')
print(f"  {'CCD':<10s}  {'thx_OCS':>10s}  {'thy_OCS':>10s}  "
      f"{'FA.x_DVCS':>11s}  {'FA.y_DVCS':>11s}   interpretation")
diff_identity = 0.0
diff_swapped  = 0.0
for name in _val_names:
    det = _det_map[name]
    ccd_mask = (det_names_all == name) & keep
    if not np.any(ccd_mask):
        continue
    idx = int(np.where(ccd_mask)[0][0])
    xp = float(aosTable['centroid_x_intra'].iloc[idx])
    yp = float(aosTable['centroid_y_intra'].iloc[idx])
    tx_ocs_deg = float(thx_deg[idx])
    ty_ocs_deg = float(thy_deg[idx])
    tx_fa = det.getTransform(PIXELS, FIELD_ANGLE)
    fa_x, fa_y = tx_fa.getMapping().applyForward(
        np.vstack([[xp], [yp]])
    )
    fa_x_deg = float(np.rad2deg(fa_x[0]))
    fa_y_deg = float(np.rad2deg(fa_y[0]))
    d_id = abs(tx_ocs_deg - fa_x_deg) + abs(ty_ocs_deg - fa_y_deg)
    d_sw = abs(tx_ocs_deg - fa_y_deg) + abs(ty_ocs_deg - fa_x_deg)
    diff_identity += d_id
    diff_swapped  += d_sw
    if d_sw < d_id:
        tag = 'thx_OCS ≈ FA.y  (CCS labelling — grid matches, no flip)'
    else:
        tag = 'thx_OCS ≈ FA.x  (DVCS labelling — intra-CCD axes FLIPPED)'
    print(f"  {name:<10s}  {tx_ocs_deg:+10.4f}  {ty_ocs_deg:+10.4f}  "
          f"{fa_x_deg:+11.4f}  {fa_y_deg:+11.4f}   {tag}")

if diff_swapped < diff_identity:
    BATOIDV2_NEEDS_AXIS_SWAP = False
    print('\n✓ thx_OCS follows the CCS convention (matches Batoid v2 grid). '
          'No intra-CCD flip.')
else:
    BATOIDV2_NEEDS_AXIS_SWAP = True
    print('\n*** thx_OCS follows the DVCS convention — the Batoid v2 '
          'interpolator is being queried with x and y swapped. ***\n'
          '    Fix: in `evaluate_zk_per_ccd_multi`, call '
          '`interp(thy_deg, thx_deg)` instead of `interp(thx_deg, thy_deg)`\n'
          '    (or re-query below with the axes swapped).')

# ---- Check (B): visual overlay ----------------------------------------
fig, axes = plt.subplots(1, len(_val_names),
                         figsize=(4.5*len(_val_names), 4.5), squeeze=False)
axes = axes[0]
for ax, name in zip(axes, _val_names):
    det = _det_map[name]
    corners = det.getCorners(FIELD_ANGLE)
    cx = [np.rad2deg(c.y) for c in corners] + [np.rad2deg(corners[0].y)]
    cy = [np.rad2deg(c.x) for c in corners] + [np.rad2deg(corners[0].x)]
    gx_rad, gy_rad = _per_ccd_field_angle_grid(det, INTRINSIC_PER_CCD_GRID_N)
    gx_deg, gy_deg = np.rad2deg(gx_rad), np.rad2deg(gy_rad)
    ccd_mask = (det_names_all == name) & keep
    d_tx = thx_deg[ccd_mask]
    d_ty = thy_deg[ccd_mask]
    ax.plot(cx, cy, 'g-', lw=1.0, label='CCD corners (CCS)')
    ax.plot(gx_deg, gy_deg, 'b.', ms=8, label=f'grid ({len(gx_deg)})')
    if len(d_tx) > 0:
        ax.plot(d_tx, d_ty, 'r.', ms=1, alpha=0.4,
                label=f'donuts thx/thy_OCS ({len(d_tx)})')
    ax.set_xlabel(r'$\theta_x$ [deg]')
    ax.set_ylabel(r'$\theta_y$ [deg]')
    ax.set_title(name)
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper right', fontsize=7)
fig.suptitle('Batoid v2 per-CCD grid vs donut (thx_OCS, thy_OCS)', fontsize=13)
fig.tight_layout()
_collect(fig)

In [ ]:
# Batoid v2 NaN diagnostic. `LinearNDInterpolator` returns NaN for any
# query outside the convex hull of its grid. `_per_ccd_field_angle_grid`
# insets by half a cell from the CCD footprint, so donuts close to each
# CCD's edge fall outside the 8×8 grid convex hull and come back NaN
# across every Zj column — the edge-artefact you're seeing in the maps.
#
# Quantify it: overall kept-donut NaN rate, per-CCD NaN rate, plus a
# focal-plane scatter marking the NaN donuts. Since the interpolator
# returns a full vector of NaNs for any out-of-hull point, we use any
# single Zj column as a sentinel and the counts apply uniformly to all
# Zj in ZJ_LIST.
_sentinel_j = ZJ_LIST[0]
bv2_nan = ~np.isfinite(Zj_bv2_um[_sentinel_j])

n_keep      = int(keep.sum())
n_nan_keep  = int((bv2_nan & keep).sum())
n_fin_keep  = n_keep - n_nan_keep
pct_nan     = 100.0 * n_nan_keep / max(n_keep, 1)
print(f'Batoid v2 NaN rate over kept donuts: '
      f'{n_nan_keep:,} / {n_keep:,}  ({pct_nan:.2f}%)  — '
      f'same count applies to every Zj in ZJ_LIST.')

# Per-CCD NaN fraction (top offenders)
_rows = []
for det in camera:
    name = det.getName()
    m = (det_names_all == name) & keep
    n = int(m.sum())
    if n == 0:
        continue
    nnan = int((m & bv2_nan).sum())
    _rows.append((name, nnan, n, nnan / n))
ccd_nan_df = (pd.DataFrame(_rows,
                           columns=['detector', 'n_nan', 'n_donuts', 'frac_nan'])
              .sort_values(by='frac_nan', ascending=False)
              .reset_index(drop=True))

print(f'\nPer-CCD Batoid v2 NaN fraction (top 20, sorted high → low):')
with pd.option_context('display.max_rows', 20,
                       'display.width', 120,
                       'display.float_format', '{:.3f}'.format):
    print(ccd_nan_df.head(20).to_string(index=False))

print(f'\nBottom 5 (CCDs with fewest NaNs):')
with pd.option_context('display.max_rows', 5,
                       'display.width', 120,
                       'display.float_format', '{:.3f}'.format):
    print(ccd_nan_df.tail(5).to_string(index=False))

print(f'\nFocal-plane-wide NaN summary (kept donuts):')
print(f'  finite  {n_fin_keep:>8,}   ({100.0 * n_fin_keep/max(n_keep,1):.2f}%)')
print(f'  NaN     {n_nan_keep:>8,}   ({pct_nan:.2f}%)')
print(f'  CCDs with any NaN: '
      f'{int((ccd_nan_df["n_nan"] > 0).sum())} / {len(ccd_nan_df)}')

# Focal-plane scatter: where the NaN donuts land.
fig, ax = plt.subplots(figsize=(8, 8))
fin_mask = keep & ~bv2_nan
nan_mask = keep &  bv2_nan
ax.scatter(fpx[fin_mask], fpy[fin_mask], c='lightgray', s=1, alpha=0.3,
           label=f'finite ({int(fin_mask.sum()):,})')
ax.scatter(fpx[nan_mask], fpy[nan_mask], c='red', s=3, alpha=0.6,
           label=f'Batoid v2 NaN ({int(nan_mask.sum()):,})')
ax.set_aspect('equal')
ax.set_xlabel('fpx [mm]')
ax.set_ylabel('fpy [mm]')
ax.set_title('Donuts where Batoid v2 intrinsic is NaN\n'
             '(out of the per-CCD half-cell-inset grid convex hull)')
ax.legend(loc='upper right', frameon=True)
ax.grid(alpha=0.3)
fig.tight_layout()
_collect(fig)

In [ ]:
# One 2×3 page per Zj:
#   Top row:    data (Z − linear),  Transposed Pipeline intrinsic,  Batoid v2
#   Bottom row: data − Transp.Pipe, data − Batoid v2,                Tr.Pipe − Batoid v2
# Top three panels share a colour scale (3 %/97 % percentiles pooled per group);
# the two data-minus-intrinsic diffs share another; the Transposed Pipeline −
# Batoid v2 panel gets its own.
n_kept = int(keep.sum())
xe = None
for j in ZJ_LIST:
    data_j  = Zj_data_linear_um[j]
    pipeT_j = Zj_pipe_trans_um[j]
    bv2_j   = Zj_bv2_um[j]

    stat_data,  xe, ye = bin_median_2d(xv, yv, data_j[keep],  xbins, ybins)
    stat_pipeT, _, _   = bin_median_2d(xv, yv, pipeT_j[keep], xbins, ybins)
    stat_bv2,   _, _   = bin_median_2d(xv, yv, bv2_j[keep],   xbins, ybins)

    stat_data_pipeT = stat_data  - stat_pipeT
    stat_data_bv2   = stat_data  - stat_bv2
    stat_pipeT_bv2  = stat_pipeT - stat_bv2

    maps = [
        (f'Data:  Z{j} − linear',                           stat_data),
        (f'Transposed Pipeline intrinsic  Z{j}',            stat_pipeT),
        (f'Batoid v2 intrinsic  Z{j}',                      stat_bv2),
        (f'Data  −  Transposed Pipeline  (Z{j})',           stat_data_pipeT),
        (f'Data  −  Batoid v2  (Z{j})',                     stat_data_bv2),
        (f'Transposed Pipeline  −  Batoid v2  (Z{j})',      stat_pipeT_bv2),
    ]
    fig = plot_fp_grid(
        maps, xe, ye,
        title=(f'Z{j}   |   rotator ≈ {ROTATOR_CENTER_DEG:.0f}° '
               f'(±{ROTATOR_TOL_DEG:.0f}°),   {n_kept:,} donuts'),
    )
    _collect(fig)

In [ ]:
# Write every collected figure (one per Zj) to a single PDF.
pdf_path = Path(PDF_OUTPUT)
pdf_path.parent.mkdir(parents=True, exist_ok=True)
with PdfPages(pdf_path) as pdf:
    for f in _pdf_figs:
        pdf.savefig(f, bbox_inches='tight')
print(f'Saved {len(_pdf_figs)} pages to {pdf_path}')